# SIC 2026 · Inteligencia Artificial
## Clase 9: Plantilla de prototipo funcional inicial y pitch

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

Este notebook es la **plantilla de trabajo de su equipo** para la clase de hoy. Lo recorren de arriba hacia abajo y al final tienen:

1. Una **Test Card** con la hipótesis que su prototipo pone a prueba.
2. Un **modelo base** entrenado y evaluado con sus datos.
3. Una **app en Gradio** que se genera sola a partir de sus columnas.
4. El modelo guardado y, si les alcanza el tiempo, publicado en Hugging Face.
5. Su **guion de pitch** con contador de palabras y tiempo.
6. La **Learning Card** y el registro de los prompts que usaron.

**Cómo usarlo:** solo tienen que cambiar las celdas marcadas con `# CAMBIEN AQUÍ`. Todo lo demás funciona solo. Si lo ejecutan sin cambiar nada, el notebook usa como ejemplo el dataset de fuga de clientes de la Clase 6, para que vean cómo debería quedar.

**Antes de empezar:** *Archivo > Guardar una copia en Drive*, y renómbrenlo como `clase09_nombre_del_equipo.ipynb`.

La guía completa de la clase (perfil del cliente, frase de valor, cinco momentos del pitch) está en el documento "Guía de prototipo y pitch" que el profesor compartió en el chat.

In [ ]:
!pip install -q gradio kagglehub huggingface_hub

import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
import sklearn
warnings.filterwarnings("ignore")
print("Listo. scikit-learn", sklearn.__version__, "| Gradio", gr.__version__)

---
# 0. Datos del equipo

In [ ]:
# CAMBIEN AQUÍ
EQUIPO = "Nombre del equipo"
INTEGRANTES = ["Integrante 1", "Integrante 2", "Integrante 3"]
FRASE_DE_VALOR = ("Ayudamos a [quién] a [lograr qué], usando [qué solución de IA], "
                  "a diferencia de [cómo se hace hoy].")

print(f"Equipo: {EQUIPO}\nIntegrantes: {', '.join(INTEGRANTES)}\n\n{FRASE_DE_VALOR}")

---
# 1. Test Card

Escriban la hipótesis **antes** de construir. El umbral es lo más importante: sin un número, no hay forma de saber si la prueba salió bien o mal. Más abajo, el notebook compara automáticamente su resultado con este umbral.

Métricas disponibles:

- **Clasificación:** `"exactitud"`, `"recall"`, `"precision"`, `"f1"` o `"roc_auc"` (recall, precision y F1 se calculan para la clase que definan como positiva).
- **Regresión:** `"r2"` (más alto es mejor) o `"mae"` (error absoluto medio, más bajo es mejor).

In [ ]:
# CAMBIEN AQUÍ
TEST_CARD = {
    "Creemos que": "se puede anticipar qué clientes se van",
    "Para verificarlo, vamos a": "entrenar un modelo con el historial de clientes de una compañía de telecomunicaciones",
    "Mediremos": "el recall de la clase 'se va' en datos de prueba",
    "Tenemos razón si": "detecta al menos 7 de cada 10",
}
METRICA = "recall"     # la métrica que van a comparar con el umbral
UMBRAL = 0.70          # el número de "Tenemos razón si"

for k, v in TEST_CARD.items():
    print(f"{k:28s} {v}")

---
# 2. Cargar los datos

Elijan **una** de las tres opciones cambiando `FUENTE`:

| FUENTE | Qué hace | Qué completar |
|---|---|---|
| `"kaggle"` | Descarga un dataset público de Kaggle con `kagglehub` (Clase 6) | `ID_KAGGLE` y `ARCHIVO` |
| `"url"` | Lee un CSV desde un link directo (por ejemplo, datos.gob.cl o GitHub) | `URL` |
| `"subir"` | Abre un cuadro para subir un CSV desde su computador | nada |
| `"ejemplo"` | Usa el dataset de fuga de clientes de la Clase 6 | nada |

In [ ]:
# CAMBIEN AQUÍ
FUENTE = "ejemplo"
ID_KAGGLE = "usuario/nombre-del-dataset"
ARCHIVO = "archivo.csv"
URL = "https://..."

def cargar_datos():
    if FUENTE == "kaggle":
        import kagglehub
        carpeta = kagglehub.dataset_download(ID_KAGGLE)
        print("Archivos en el dataset:", os.listdir(carpeta))
        return pd.read_csv(os.path.join(carpeta, ARCHIVO))
    if FUENTE == "url":
        return pd.read_csv(URL)
    if FUENTE == "subir":
        from google.colab import files
        subido = files.upload()
        return pd.read_csv(list(subido)[0])
    # Ejemplo: fuga de clientes (Clase 6)
    try:
        import kagglehub
        carpeta = kagglehub.dataset_download("blastchar/telco-customer-churn")
        return pd.read_csv(os.path.join(carpeta, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))
    except Exception:
        return pd.read_csv("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")

datos = cargar_datos()
print("Forma:", datos.shape)
datos.head()

In [ ]:
# Revisión rápida: tipos y faltantes
resumen = pd.DataFrame({"tipo": datos.dtypes.astype(str), "faltantes": datos.isna().sum(),
                        "valores distintos": datos.nunique()})
resumen

---
# 3. Definir el problema

Elijan la **variable objetivo** (lo que el modelo predice) y las **características** que usará. Recuerden:

- Para la app, conviene usar **pocas características** (entre 4 y 10) que una persona pueda ingresar fácilmente.
- **No incluyan identificadores** (como `customerID`) ni columnas que contengan la respuesta (fuga de datos, Clase 4).
- Si una columna numérica viene como texto (como `TotalCharges` en la Clase 6), el notebook intenta convertirla sola.

In [ ]:
# CAMBIEN AQUÍ
OBJETIVO = "Churn"
CARACTERISTICAS = ["tenure", "MonthlyCharges", "Contract", "InternetService", "PaymentMethod", "TechSupport"]
CLASE_POSITIVA = "Yes"     # solo para clasificación: la clase que les importa detectar (para recall, precision y F1)

In [ ]:
df = datos[CARACTERISTICAS + [OBJETIVO]].copy()

# Convertir a número las columnas de texto que en realidad son números
for col in CARACTERISTICAS:
    if df[col].dtype == object:
        convertida = pd.to_numeric(df[col], errors="coerce")
        if convertida.notna().mean() > 0.95:
            df[col] = convertida
            print(f"'{col}' convertida a número")

df = df.dropna(subset=[OBJETIVO])
NUMERICAS = [c for c in CARACTERISTICAS if pd.api.types.is_numeric_dtype(df[c])]
CATEGORICAS = [c for c in CARACTERISTICAS if c not in NUMERICAS]

# ¿Clasificación o regresión? Si el objetivo es texto o tiene pocos valores distintos, es clasificación
ES_CLASIFICACION = (not pd.api.types.is_numeric_dtype(df[OBJETIVO])) or df[OBJETIVO].nunique() <= 10

print("Numéricas:", NUMERICAS)
print("Categóricas:", CATEGORICAS)
print("Tipo de problema:", "clasificación" if ES_CLASIFICACION else "regresión")
if ES_CLASIFICACION:
    print(df[OBJETIVO].value_counts(normalize=True).round(3))

---
# 4. Modelo base

El objetivo de hoy **no es el mejor modelo**: es tener uno que funcione. Usamos el mismo esquema de la Clase 6:

- Un `Pipeline` que rellena faltantes, escala las numéricas y aplica one-hot a las categóricas.
- Una **línea base** (`Dummy`) para saber cuál es el piso.
- Un modelo simple: **regresión logística** si es clasificación, **regresión lineal** si es regresión.

Si quieren probar otro modelo, cambien `MODELO`. Por ejemplo: `RandomForestClassifier(random_state=42)`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression

# CAMBIEN AQUÍ (opcional)
MODELO = LogisticRegression(max_iter=3000, class_weight="balanced") if ES_CLASIFICACION else LinearRegression()

preprocesador = ColumnTransformer([
    ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), NUMERICAS),
    ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAS),
])

X = df[CARACTERISTICAS]
y = df[OBJETIVO]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if ES_CLASIFICACION else None)

base = Pipeline([("prep", preprocesador), ("modelo", DummyClassifier() if ES_CLASIFICACION else DummyRegressor())]).fit(X_train, y_train)
modelo = Pipeline([("prep", preprocesador), ("modelo", MODELO)]).fit(X_train, y_train)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)

In [ ]:
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score, roc_auc_score,
                             r2_score, mean_absolute_error, classification_report, ConfusionMatrixDisplay)

def evaluar(m):
    pred = m.predict(X_test)
    if ES_CLASIFICACION:
        binaria = y.nunique() == 2
        res = {"exactitud": accuracy_score(y_test, pred)}
        if binaria:
            res["recall"] = recall_score(y_test, pred, pos_label=CLASE_POSITIVA, zero_division=0)
            res["precision"] = precision_score(y_test, pred, pos_label=CLASE_POSITIVA, zero_division=0)
            res["f1"] = f1_score(y_test, pred, pos_label=CLASE_POSITIVA, zero_division=0)
            idx = list(m.classes_).index(CLASE_POSITIVA)
            res["roc_auc"] = roc_auc_score((y_test == CLASE_POSITIVA).astype(int), m.predict_proba(X_test)[:, idx])
        else:
            res["f1"] = f1_score(y_test, pred, average="macro", zero_division=0)
        return res
    return {"r2": r2_score(y_test, pred), "mae": mean_absolute_error(y_test, pred)}

resultados = pd.DataFrame({"Línea base": evaluar(base), "Modelo base": evaluar(modelo)}).round(3)
resultados

In [ ]:
if ES_CLASIFICACION:
    print(classification_report(y_test, modelo.predict(X_test), digits=3))
    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test, cmap="Blues")
    plt.title("Matriz de confusión del modelo base"); plt.show()
else:
    pred = modelo.predict(X_test)
    plt.scatter(y_test, pred, alpha=0.4); plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "--", color="gray")
    plt.xlabel("Valor real"); plt.ylabel("Predicción"); plt.title("Predicción frente a valor real"); plt.show()

## ¿Se cumplió la Test Card?

In [ ]:
valor = resultados.loc[METRICA, "Modelo base"]
mas_bajo_es_mejor = METRICA == "mae"
cumple = valor <= UMBRAL if mas_bajo_es_mejor else valor >= UMBRAL

print(f"Métrica: {METRICA} = {valor:.3f} | Umbral: {UMBRAL}")
print("Tenemos razón: la hipótesis se cumple." if cumple else
      "Todavía no: la hipótesis no se cumple con este modelo. Es información valiosa para la Learning Card.")

---
# 5. Predicción para un caso nuevo

Antes de armar la app, prueben la **función de predicción sola** (la lección de la Clase 6). Cambien los valores del caso para probar distintas situaciones.

In [ ]:
# Valores de referencia para la app: mediana de las numéricas y categorías de las categóricas
REFERENCIAS = {
    "numericas": {c: {"min": float(df[c].min()), "max": float(df[c].max()), "valor": float(df[c].median())} for c in NUMERICAS},
    "categoricas": {c: sorted(df[c].dropna().astype(str).unique().tolist()) for c in CATEGORICAS},
}

def predecir(*valores):
    caso = pd.DataFrame([dict(zip(CARACTERISTICAS, valores))])
    if ES_CLASIFICACION:
        probas = modelo.predict_proba(caso)[0]
        return {str(c): float(p) for c, p in zip(modelo.classes_, probas)}
    return float(modelo.predict(caso)[0])

# CAMBIEN AQUÍ: un caso de prueba, en el mismo orden de CARACTERISTICAS
caso_prueba = X_test.iloc[0].tolist()
print("Caso:", {k: (v.item() if hasattr(v, "item") else v) for k, v in zip(CARACTERISTICAS, caso_prueba)})
print("Predicción:", predecir(*caso_prueba))

---
# 6. App en Gradio (se genera sola)

La app arma un componente por cada característica:

- **Numéricas:** un `gr.Number` con la mediana como valor inicial y una ayuda con el rango observado.
- **Categóricas:** un `gr.Dropdown` con las categorías que existen en los datos.

La salida es `gr.Label` (probabilidad de cada clase) en clasificación, o `gr.Number` en regresión. Cambien `TITULO` y `DESCRIPCION`.

In [ ]:
# CAMBIEN AQUÍ
TITULO = "Riesgo de fuga de clientes"
DESCRIPCION = "Prototipo inicial. Ingresa los datos del cliente para estimar la probabilidad de que deje la compañía."

def construir_entradas():
    entradas = []
    for c in CARACTERISTICAS:
        if c in NUMERICAS:
            r = REFERENCIAS["numericas"][c]
            entradas.append(gr.Number(value=r["valor"], label=c, info=f"Rango en los datos: {r['min']:g} a {r['max']:g}"))
        else:
            opciones = REFERENCIAS["categoricas"][c]
            entradas.append(gr.Dropdown(opciones, value=opciones[0], label=c))
    return entradas

with gr.Blocks(title=TITULO) as app:
    gr.Markdown(f"# {TITULO}\n{DESCRIPCION}")
    with gr.Row():
        with gr.Column():
            entradas = construir_entradas()
            boton = gr.Button("Predecir", variant="primary")
        with gr.Column():
            salida = gr.Label(label="Probabilidad") if ES_CLASIFICACION else gr.Number(label="Predicción")
            gr.Markdown(f"*Equipo {EQUIPO}. Modelo base, prototipo inicial: no usar para decisiones reales.*")
    boton.click(predecir, inputs=entradas, outputs=salida)

app.launch()

**Prueben la app con 3 casos distintos** y anoten qué predice y si tiene sentido. Lo usarán en la demo y en la Learning Card.

| Caso | Qué ingresaron | Qué predijo | ¿Tiene sentido? |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

---
# 7. Guardar y publicar (nivel avanzado)

Guardamos el modelo con todo lo que necesita la app, y generamos `app.py` y `requirements.txt` en la carpeta `space_prototipo/`, igual que en la Clase 6. La app de esa carpeta es **genérica**: lee las características y referencias desde el pickle, así que funciona con cualquier proyecto sin cambiar código.

In [ ]:
os.makedirs("space_prototipo", exist_ok=True)
paquete = {"modelo": modelo, "caracteristicas": CARACTERISTICAS, "numericas": NUMERICAS,
           "referencias": REFERENCIAS, "es_clasificacion": ES_CLASIFICACION,
           "titulo": TITULO, "descripcion": DESCRIPCION, "equipo": EQUIPO,
           "version_sklearn": sklearn.__version__}
with open("space_prototipo/modelo.pkl", "wb") as f:
    pickle.dump(paquete, f)

codigo_app = '''import pickle
import pandas as pd
import gradio as gr

with open("modelo.pkl", "rb") as f:
    p = pickle.load(f)
modelo, CARACT = p["modelo"], p["caracteristicas"]

def predecir(*valores):
    caso = pd.DataFrame([dict(zip(CARACT, valores))])
    if p["es_clasificacion"]:
        return {str(c): float(x) for c, x in zip(modelo.classes_, modelo.predict_proba(caso)[0])}
    return float(modelo.predict(caso)[0])

with gr.Blocks(title=p["titulo"]) as app:
    gr.Markdown("# " + p["titulo"] + "\\n" + p["descripcion"])
    with gr.Row():
        with gr.Column():
            entradas = []
            for c in CARACT:
                if c in p["numericas"]:
                    r = p["referencias"]["numericas"][c]
                    entradas.append(gr.Number(value=r["valor"], label=c, info=f"Rango en los datos: {r['min']:g} a {r['max']:g}"))
                else:
                    op = p["referencias"]["categoricas"][c]
                    entradas.append(gr.Dropdown(op, value=op[0], label=c))
            boton = gr.Button("Predecir", variant="primary")
        with gr.Column():
            salida = gr.Label(label="Probabilidad") if p["es_clasificacion"] else gr.Number(label="Predicción")
            gr.Markdown("*Equipo " + p["equipo"] + ". Prototipo inicial: no usar para decisiones reales.*")
    boton.click(predecir, inputs=entradas, outputs=salida)

app.launch()
'''
with open("space_prototipo/app.py", "w", encoding="utf-8") as f:
    f.write(codigo_app)
with open("space_prototipo/requirements.txt", "w") as f:
    f.write(f"scikit-learn=={sklearn.__version__}\npandas\n")
print("Archivos listos:", os.listdir("space_prototipo"))

Para publicar, sigan los pasos de la Parte 4 de la Clase 6: token de escritura guardado en los secretos de Colab como `HF_TOKEN`, y luego ejecuten la celda siguiente. Si prefieren la forma web, suban los tres archivos de `space_prototipo/` a un Space nuevo con SDK Gradio.

In [ ]:
# Ejecuten esta celda solo si van a publicar
PUBLICAR = False                      # CAMBIEN AQUÍ a True para publicar
NOMBRE_SPACE = "prototipo-mi-equipo"  # CAMBIEN AQUÍ: sin espacios ni tildes

if PUBLICAR:
    from huggingface_hub import HfApi, login
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    api = HfApi()
    repo_id = f"{api.whoami()['name']}/{NOMBRE_SPACE}"
    api.create_repo(repo_id=repo_id, repo_type="space", space_sdk="gradio", exist_ok=True)
    api.upload_folder(folder_path="space_prototipo", repo_id=repo_id, repo_type="space")
    print(f"Publicado en: https://huggingface.co/spaces/{repo_id}")
else:
    print("Publicación desactivada. Cambien PUBLICAR a True cuando estén listos.")

---
# 8. Guion de pitch

Escriban su guion en los cinco momentos: **150 a 250 palabras** en total. La celda cuenta las palabras y estima el tiempo, a un ritmo de unas 130 palabras por minuto.

| Momento | Qué responde |
|---|---|
| Gancho | Una frustración, un dato que impacte o una historia |
| Problema | Quién tiene el problema, qué intenta lograr y qué le frustra |
| Solución | Qué hace el prototipo y en qué se diferencia |
| Evidencia | Con qué datos se entrenó y qué tan bien funciona, en lenguaje simple |
| Cierre | Visión, por qué este equipo y una invitación concreta |

**Regla práctica:** si una frase solo la entiende alguien que tomó este curso, va a la demo, no al pitch.

In [ ]:
# CAMBIEN AQUÍ (el ejemplo está resumido a propósito: el contador les dirá que está corto)
GUION = {
    "Gancho": "En la compañía que estudiamos, más de uno de cada cuatro clientes se fue en un solo mes.",
    "Problema": "Los ejecutivos de retención tienen miles de clientes y no saben a quién llamar primero.",
    "Solución": "Nuestro prototipo estima el riesgo de fuga de cada cliente y dice a quién contactar primero.",
    "Evidencia": "Lo entrenamos con miles de clientes reales y, con clientes que nunca vio, detecta a la mayoría de los que se van.",
    "Cierre": "Queremos que ninguna compañía se entere tarde. Buscamos una empresa que nos deje probarlo un mes.",
}

total = 0
for momento, texto in GUION.items():
    n = len(texto.split()); total += n
    print(f"{momento:10s} {n:3d} palabras")
segundos = total / 130 * 60
print(f"\nTotal: {total} palabras | Tiempo estimado: {segundos:.0f} segundos")
if total < 150:
    print("Está corto: agreguen detalle, sobre todo en Problema y Evidencia.")
elif total > 250:
    print("Está largo: corten. No se trata de hablar más rápido.")
else:
    print("Largo adecuado. Léanlo en voz alta con cronómetro.")

In [ ]:
# Para la evidencia del pitch: el resultado de su modelo, en una frase simple
if ES_CLASIFICACION and "recall" in resultados.index:
    r = resultados.loc["recall", "Modelo base"]
    print(f"Con datos que nunca vio, el modelo detecta {round(r * 10)} de cada 10 casos de '{CLASE_POSITIVA}'.")
elif ES_CLASIFICACION:
    print(f"Con datos que nunca vio, el modelo acierta {resultados.loc['exactitud', 'Modelo base']:.0%} de las veces.")
else:
    print(f"En promedio, la predicción se equivoca en {resultados.loc['mae', 'Modelo base']:.2f} unidades.")

---
# 9. Learning Card y registro de prompts

## Learning Card

Complétenla después de presentar (doble clic para editar):

| | |
|---|---|
| **Creíamos que** | |
| **Observamos que** | (la métrica real y lo que dijo el curso en la devolución) |
| **De esto aprendimos que** | |
| **Por lo tanto, vamos a** | Perseverar / Pivotar / Archivar |

## Prompts de IA generativa que usamos

Anoten los prompts principales: van al informe final, en la sección de metodología.

| Herramienta | Prompt | ¿Para qué sirvió? | ¿Lo modificamos? |
|---|---|---|---|
| | | | |
| | | | |

## Tarea (antes del jueves 5 de noviembre)

Suban a la carpeta de su equipo: este notebook con la Test Card, la Learning Card y el guion mejorado, y el link al prototipo (este notebook o su Space).